# Figure 2 — Publication trend of the included articles by dominant thematic cluster

Reproduces the stacked bar chart for the systematic review (CNKI, 96 included articles, 2010–2026).

**Input:** `screening_v9_PRISMAequitycoding.xlsx`, sheet *Screening & equity* (header on row 2).
An article is in the corpus if `YOUR DECISION = Include` **and** `Full text retrieved = Y`.
The **dominant cluster** is the first label in *Thematic cluster (manual)* (labels separated by `/`).

**Output:** `fig2_publication_trend_96.png` (300 dpi), `.pdf` and the data table `_data.csv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

INPUT = "screening_v9_PRISMAequitycoding.xlsx"   # path to the screening workbook
SHEET = "Screening & equity"
OUT = "fig2_publication_trend_96_v4"
SEARCH_NOTE = "2026 up to the search date (28 October 2026)"

In [ ]:
df = pd.read_excel(INPUT, sheet_name=SHEET, header=1)

dec = df["YOUR DECISION (Include/Exclude)"].astype(str).str.strip().str.lower()
ft  = df["Full text retrieved (Y/N)"].astype(str).str.strip().str.upper()
corpus = df[(dec == "include") & (ft == "Y")].copy()

# Dominant cluster = first label of the manual cluster column
corpus["cluster"] = (corpus["Thematic cluster (manual)"].astype(str)
                     .str.split("/").str[0].str.strip())

# Publication year. ID 49 (Teng, 2025) is an online-first article with no year in the CNKI export;
# it was published online on 3 July 2025, so it is assigned to 2025.
corpus["year"] = pd.to_numeric(corpus["Year"], errors="coerce")
corpus.loc[corpus["ID"] == 49, "year"] = 2025
assert corpus["year"].notna().all(), corpus.loc[corpus["year"].isna(), ["ID", "Title (Chinese)"]]
corpus["year"] = corpus["year"].astype(int)

print("Included articles:", len(corpus))
print(corpus["cluster"].value_counts())

In [ ]:
# Cluster order = stacking order (bottom -> top). Colours: Okabe-Ito based, the same colours as the
# earlier version of the figure for the six original clusters; 'Legal & legislative framework' (#7F3C8D) is new.
# This order was checked for colour-blind separation of adjacent segments (palette validator: all checks pass).
CLUSTERS = [
    ("Policy & governance",            "#0072B2"),
    ("Agriculture & food security",    "#009E73"),
    ("Legal & legislative framework",  "#7F3C8D"),
    ("Science & technology support",   "#E69F00"),
    ("Urban resilience & planning",    "#CC79A7"),
    ("Financing mechanisms",           "#56B4E9"),
    ("Social equity & participation",  "#D55E00"),
]
unknown = set(corpus["cluster"]) - {c for c, _ in CLUSTERS}
assert not unknown, f"Unexpected cluster labels: {unknown}"

years = list(range(corpus["year"].min(), corpus["year"].max() + 1))
counts = (corpus.pivot_table(index="year", columns="cluster", values="ID", aggfunc="count", fill_value=0)
                .reindex(index=years, fill_value=0)
                .reindex(columns=[c for c, _ in CLUSTERS], fill_value=0))
counts

In [ ]:
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10})
fig, ax = plt.subplots(figsize=(11.5, 5.6), dpi=150)

bottom = pd.Series(0, index=years)
for name, colour in CLUSTERS:
    vals = counts[name]
    ax.bar(years, vals, bottom=bottom, width=0.7, color=colour,
           edgecolor="white", linewidth=1.2, label=name, zorder=3)   # white gap between stacked segments
    bottom += vals

totals = counts.sum(axis=1)
for y, t in totals.items():
    if t > 0:
        ax.text(y, t + 0.15, str(t), ha="center", va="bottom", fontsize=9, fontweight="bold", color="#222222")

ymax = int(totals.max()) + 5

def milestone(x, label, y_text):
    ax.annotate(label, xy=(x, totals[x] + 0.9), xytext=(x, y_text), ha="center", va="bottom", fontsize=8.5,
                arrowprops=dict(arrowstyle="-|>", color="#333333", lw=0.8),
                bbox=dict(boxstyle="round,pad=0.35", fc="#FBF8EF", ec="#333333", lw=0.6))

milestone(2013, "First National Climate Change\nAdaptation Strategy (2013)", ymax - 3.2)
milestone(2022, "National Climate Change Adaptation\nStrategy 2035 (2022)", ymax - 3.2)

ax.set_xticks(years)
ax.set_xticklabels([f"{y}*" if y == years[-1] else str(y) for y in years], rotation=45)
ax.set_ylim(0, ymax)
ax.set_xlim(years[0] - 0.6, years[-1] + 0.6)
ax.set_xlabel("Year of publication")
ax.set_ylabel("Number of articles")
ax.yaxis.grid(True, color="#E5E5E5", linewidth=0.8)
ax.set_axisbelow(True)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)

ax.set_title(f"Publication trend of included articles by dominant thematic cluster, {years[0]}–{years[-1]}",
             loc="left", fontsize=13, fontweight="bold", pad=24)
ax.text(0, 1.025, f"n = {len(corpus)} articles · China National Knowledge Infrastructure (CNKI) · * {SEARCH_NOTE}",
        transform=ax.transAxes, fontsize=8.5, color="#555555")

ax.legend(title="Dominant thematic cluster", loc="upper left", bbox_to_anchor=(1.01, 1.0),
          frameon=False, fontsize=9, title_fontsize=9.5, reverse=True)
fig.tight_layout()
fig.savefig(OUT + ".png", dpi=300, bbox_inches="tight")
fig.savefig(OUT + ".pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Data table behind the figure (for checking, or as a supplementary table)
table = counts.copy()
table["Total"] = table.sum(axis=1)
table.loc["Total"] = table.sum()
table.to_csv(OUT + "_data.csv")
table